In [0]:
import dlt
from pyspark.sql.functions import current_timestamp, col, regexp_extract, input_file_name, to_date,explode
landing_path = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/fred_raw/data/"
schema_path  = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/_schemas/fred_raw_json/"

@dlt.table(
    name="bronze_fred_raw_json",
    comment="Raw FRED API payloads as JSON (one row per file)"
)
def bronze_fred_raw_json():
    return (
        spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "json")
          .option("cloudFiles.schemaLocation", schema_path)
          .option("cloudFiles.includeExistingFiles", "true")
          .option("recursiveFileLookup", "true")
          .option("multiLine", "true")
          .load(landing_path)
          .withColumn("_ingest_ts", current_timestamp())
          .withColumn("_source_file", col("_metadata.file_path"))
          .withColumn("series_id", regexp_extract(col("_metadata.file_path"), r"/data/([^/]+)/", 1))
    )

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, DoubleType
from pyspark.sql.functions import explode, col, to_date, input_file_name, current_timestamp

# ---- Google Trends paths ----
gt_landing_path = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/google_trends_raw/data/"
gt_schema_path  = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/_schemas/google_trends_raw/"

# ---- Explicit schema (fix for Auto Loader inference error) ----
gt_schema = StructType([
    StructField("keyword", StringType(), True),
    StructField("geo", StringType(), True),
    StructField("timeframe", StringType(), True),
    StructField("run_ts", StringType(), True),
    StructField("observations", ArrayType(StructType([
        StructField("date", StringType(), True),
        StructField("value", DoubleType(), True),
    ])), True)
])

@dlt.table(
  name="bronze_google_trends_raw",
  comment="Google Trends raw JSON exploded to one row per (keyword, date)"
)
def bronze_google_trends_raw():

    raw = (
        spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "json")
          .option("cloudFiles.schemaLocation", gt_schema_path)
          .option("cloudFiles.includeExistingFiles", "true")
          .option("recursiveFileLookup", "true")
          .schema(gt_schema)  # ✅ explicit schema
          .load(gt_landing_path)
    )

    return (
    raw.withColumn("obs", explode(col("observations")))
       .select(
           col("keyword"),
           col("geo"),
           col("timeframe"),
           to_date(col("obs.date")).alias("date"),
           col("obs.value").cast("double").alias("value"),
           col("run_ts"),
           col("_metadata.file_path").alias("source_file"),   # ✅ UC-safe
           current_timestamp().alias("ingest_ts")
       )
)

In [0]:
from pyspark.sql.functions import col, current_timestamp, decode


landing_path = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/wits_tariff_raw/data/"
schema_path  = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/_schemas/wits_tariff_bronze_binary/"  # NEW

@dlt.table(
  name="bronze_wits_tariff_raw",
  comment="Raw WITS SDMX JSON — one row per file (binaryFile)"
)
def bronze_wits_tariff_raw():

  df = (
    spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "binaryFile")
      .option("cloudFiles.schemaLocation", schema_path)
      .option("cloudFiles.includeExistingFiles", "true")
      .option("recursiveFileLookup", "true")
      .load(landing_path)
  )

  return df.select(
    col("path").alias("source_file"),
    decode(col("content"), "UTF-8").alias("raw_payload"),
    col("length").alias("raw_len"),
    current_timestamp().alias("ingest_ts")
  )